# Route definition 

In [3]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]


# IVA calculations



In [4]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_gastos_2018= pd.read_csv(
    RAW_DIR / "gastoshogar_2018.csv",
    dtype={
        "folioviv": str,
        "foliohog": str,
        "clave": str
    },
    low_memory=False
)

# ==========================================================
# Limpiar variables
# ==========================================================

df_gastos_2018["gasto_tri"] = pd.to_numeric(
    df_gastos_2018["gasto_tri"],
    errors="coerce"
).fillna(0)

# Sólo gastos monetarios
df_gastos_2018 = df_gastos_2018[df_gastos_2018["tipo_gasto"] == "G1"]

# Sólo compras formales
informal_locs = [1, 2, 3, 17]

df_gastos_2018["es_formal"] = (
    ~df_gastos_2018["lugar_comp"].isin(informal_locs)
).astype(int)

# ==========================================================
# Identificar productos con IVA
# ==========================================================

CLAVES_IVA_CERO_2018 = [
    # Cereales y derivados (básicos)
    'A001', 'A002', 'A003', 'A004', 'A007', 'A008', 'A019',
    # Carnes (res, puerco, pollo, pescado)
    'A025', 'A026', 'A027', 'A028', 'A029', 'A030', 'A031', 'A032', 'A033', 'A034', 'A035', 'A036', 'A037',
    'A038', 'A039', 'A040', 'A041', 'A042', 'A043', 'A044', 'A045', 'A046',
    'A057', 'A058', 'A059', 'A060', 'A061',
    'A066', 'A067', 'A068', 'A069', 'A070',
    # Leche y huevo
    'A075', 'A076', 'A077', 'A078', 'A079', 'A080', 'A081', 'A093',
    # Aceites
    'A095', 'A096',
    # Verduras, frutas y legumbres
    'A101', 'A102', 'A103', 'A104', 'A107', 'A108', 'A109', 'A110', 'A111', 'A112',
    'A113', 'A114', 'A115', 'A116', 'A117', 'A118', 'A119', 'A120', 'A121', 'A122',
    'A123', 'A124', 'A125', 'A126', 'A127', 'A128', 'A129', 'A130', 'A131',
    'A137', 'A138', 'A139', 'A140', 'A141',
    'A147', 'A148', 'A149', 'A150', 'A151', 'A152', 'A153', 'A154', 'A155', 'A156',
    'A157', 'A158', 'A159', 'A160', 'A161', 'A162', 'A163', 'A164', 'A165', 'A166',
    'A167', 'A168', 'A169', 'A170',
    # Azúcar y sal
    'A173', 'A191',
    # Agua natural
    'A215',
    # Transporte público
    'B001', 'B002', 'B003', 'B004', 'B005', 'B006', 'B007',
    # Educación (colegiaturas y servicios)
    'E001', 'E002', 'E003', 'E004', 'E005', 'E006', 'E007', 'E008', 'E009', 'E010', 'E011',
    # Libros, periódicos, revistas
    'E022', 'E023', 'E024',
    # Renta de vivienda
    'G101', 'G102', 'G103', 'G104', 'G105', 'G106',
    # Servicios médicos y medicamentos recetados
    'J007', 'J008', 'J016', 'J017', 'J018', 'J039',
    'J020', 'J021', 'J022', 'J023', 'J024', 'J025', 'J026', 'J027', 'J028', 'J029',
    'J030', 'J031', 'J032', 'J033', 'J034', 'J035',
    'J044', 'J045', 'J046', 'J047', 'J048', 'J049', 'J050', 'J051', 'J052', 'J053',
    'J054', 'J055', 'J056', 'J057', 'J058', 'J059',
]

df_gastos_2018["lleva_iva"] = (
    ~df_gastos_2018["clave"].isin(CLAVES_IVA_CERO_2018)
).astype(int)

# ==========================================================
# Separar gasto gravado y no gravado
# ==========================================================

df_gastos_2018["gasto_iva"] = np.where(
    (df_gastos_2018["lleva_iva"] == 1) &
    (df_gastos_2018["es_formal"] == 1),
    df_gastos_2018["gasto_tri"],
    0
)

df_gastos_2018["gasto_no_iva"] = np.where(
    df_gastos_2018["lleva_iva"] == 0,
    df_gastos_2018["gasto_tri"],
    0
)

# ==========================================================
# Gasto en electricidad y en gasolina 
# ==========================================================

df_gastos_2018["gasto_gas"] = np.where(
    df_gastos_2018["clave"].isin([
        "F009",  # Diésel
        "F007",  # Gasolina magna, regular (bajo octanaje)
        "F008"  # Gasolina premium (alto octanaje)
    ]),
    df_gastos_2018["gasto_tri"],
    0
)

df_gastos_2018["electricity"] = np.where(
    df_gastos_2018["clave"] == "R001",	#Energía eléctrica
    df_gastos_2018["gasto_tri"], 
    0
)

# ==========================================================
# IVA pagado (16% incluido en el precio)
# ==========================================================

FACTOR_IVA = 16 / 116

df_gastos_2018["iva_pagado"] = df_gastos_2018["gasto_iva"] * FACTOR_IVA

# ==========================================================
# Resumen por hogar
# ==========================================================

iva_hogar_2018 = (
    df_gastos_2018
    .groupby(["folioviv", "foliohog"], as_index=False)
    .agg(
        gasto_con_iva=("gasto_iva", "sum"),
        gasto_sin_iva=("gasto_no_iva", "sum"),
        iva_pagado=("iva_pagado", "sum"),
        gasto_total=("gasto_tri", "sum"), 
        gasto_gas = ("gasto_gas", "sum"), 
        gasto_electricity = ("electricity", "sum")
    )
    .assign(year=2018)
)

iva_hogar_2018.head()

,folioviv,foliohog,gasto_con_iva,gasto_sin_iva,iva_pagado,gasto_total,gasto_gas,gasto_electricity,year
0,0100013601,1,15350.12,2429.93,2117.257931,18551.47,7200.0,600.0,2018
1,0100013602,1,36716.03,5914.78,5064.280000,47809.97,3600.0,450.0,2018
2,0100013603,1,65681.95,26012.12,9059.579310,92851.21,4500.0,375.0,2018
3,0100013604,1,10663.63,2251.38,1470.845517,19340.06,6000.0,480.0,2018
4,0100013606,1,6611.36,4338.12,911.911724,13605.03,0.0,300.0,2018


In [5]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_gastos_2024= pd.read_csv(
    RAW_DIR / "gastoshogar_2024.csv",
    dtype={
        "folioviv": str,
        "foliohog": str,
        "clave": str
    },
    low_memory=False
)

# ==========================================================
# Limpiar variables
# ==========================================================

df_gastos_2024["gasto_tri"] = pd.to_numeric(
    df_gastos_2024["gasto_tri"],
    errors="coerce"
).fillna(0)

# Sólo gastos monetarios
df_gastos_2024 = df_gastos_2024[df_gastos_2024["tipo_gasto"] == "G1"]

# Sólo compras formales
informal_locs = [1, 2, 3, 17]

df_gastos_2024["es_formal"] = (
    ~df_gastos_2024["lugar_comp"].isin(informal_locs)
).astype(int)

# ==========================================================
# Identificar productos con IVA
# ==========================================================

CLAVES_IVA_CERO_2024 = [
    # 01: Alimentos y bebidas no alcohólicas (tasa 0% para básicos)
    '011111', '011112', '011113',  # Arroz, maíz, otros cereales
    '011121', '011122',           # Harinas de maíz y trigo
    '011131', '011132',           # Tortillas
    '011133',                     # Pan (general)
    '011191', '011192', '011193', # Tostadas, masa
    '011210',                     # Animales vivos
    '011221', '011222', '011223', '011224', '011225', '011226', '011227',  # Carnes de res
    '011228', '011229', '01122A', '01122B', '01122C',                     # Carnes de puerco
    '01122D',                     # Cordero y cabra
    '01122F', '01122G',           # Pollo y otras aves
    '01122H', '01122I',           # Otras carnes e insectos
    '011241', '011242', '011243', '011244', # Vísceras
    '011245', '011246', '011247',           # Otras partes de pollo y res
    '011310',                     # Pescado fresco
    '011331',                     # Filetes empanizados
    '011341', '011342', '011343', # Camarón y mariscos
    '011370',                     # Huevas y despojos de mariscos
    '011411', '011412', '011420', # Leche fresca y descremada
    '011432',                     # Leche en polvo
    '011433',                     # Crema
    '011441', '011442',           # Leche de soya y otras
    '011451', '011452', '011454', '011455', '011456',  # Quesos (fresco, amarillo, Oaxaca, añejo, otros)
    '011460',                     # Yogurt
    '011471',                     # Bebidas de leche
    '011481', '011482',           # Huevo de gallina y otros
    '011490',                     # Otros lácteos
    '011511', '011512',           # Aceite de oliva y otros
    '011520',                     # Mantequilla
    '011530',                     # Margarina
    '011591',                     # Manteca de puerco
    '011592',                     # Chicharrón
    '011593',                     # Otras grasas animales
    '011611', '011612', '011613', '011614', '011615', '011616', '011617',  # Frutas tropicales
    '011621', '011622', '011623', '011624',  # Cítricos
    '011631', '011632', '011633', '011634', '011635',  # Frutas de hueso
    '011641', '011642',           # Fresas y bayas
    '011651', '011652', '011653', '011654',  # Melón, sandía, uva, otras
    '011660',                     # Fruta congelada
    '011670',                     # Frutos secos
    '011680',                     # Nueces
    '011711', '011712', '011713',  # Hortalizas de hoja
    '011721', '011722', '011723', '011724', '011725', '011726',  # Calabaza, chayote, chiles
    '011727',                     # Jitomate
    '011728',                     # Pepino
    '011729',                     # Tomate verde
    '01172A',                     # Otras hortalizas frutales
    '011731', '011732', '011733',  # Chícharo, ejote, otras legumbres
    '011741', '011742', '011743', '011744', '011745', '011746', '011747', '011748', '011749',  # Otras verduras
    '011751',                     # Papa
    '011752',                     # Betabel
    '011753',                     # Jícama
    '011754',                     # Plátano macho
    '011755',                     # Otros tubérculos
    '011761', '011762', '011763', '011764',  # Frijol, lentejas, garbanzo, otras legumbres
    '011770',                     # Hortalizas secas
    '011781', '011782',           # Verduras y legumbres congeladas
    '011791',                     # Harina de papa
    '011793',                     # Chiles secos
    '011796',                     # Papas fritas (tasa 0% si son básicas)
    '011797',                     # Frijol procesado
    '011810',                     # Azúcar
    '011820',                     # Sustitutos de azúcar
    '011831',                     # Miel de abeja
    '011832',                     # Otras mieles
    '011833',                     # Mermeladas
    '011841', '011842', '011843',  # Cremas de cacahuate y otras
    '011861',                     # Helados
    '011862',                     # Hielo
    '011937',                     # Sal
    '012501',                     # Agua natural embotellada
    # 02: Bebidas alcohólicas y tabaco (generalmente 16%, pero algunos pueden tener 0%)
    # 03: Vestido y calzado (generalmente 16%)
    # 04: Vivienda (tasa 0% para rentas)
    '046261', '046262', '046263', '046264', '046265', '046266',  # Renta y estimaciones de alquiler
    # 05: Mobiliario (generalmente 16%)
    # 06: Salud (tasa 0% para medicinas y servicios médicos)
    '061111', '061112', '061113', '061114', '061115', '061116', '061117',
    '061118', '061119', '06111A', '06111B', '06111C', '06111D', '06111E',
    '06111F', '06111G', '06111H', '06111I', '06111J', '06111K', '06111L',  # Medicamentos
    '061120',                     # Productos homeopáticos y naturistas
    '061211', '061212',           # Pruebas de embarazo, termómetros, glucómetros
    '061221',                     # Preservativos
    '061222',                     # Parches de nicotina
    '061230',                     # Productos de tratamiento (jeringas, etc.)
    '061311',                     # Lentes
    '062110',                     # Vacunación
    '062191', '062192', '062193', '062194', '062195', '062196', '062197', '062198',  # Consultas médicas
    '062210',                     # Consultas dentales
    '062291', '062292',           # Ortodoncia y prótesis dentales
    '062311', '062312', '062313',  # Servicios curativos y rehabilitación
    '062321', '062322', '062323',  # Servicios de enfermería y asistencia
    '063101', '063102', '063103', '063104', '063105', '063106',  # Hospitalización
    '063200',                     # Hospitalización en salud mental
    '064101', '064102', '064103',  # Análisis clínicos y estudios
    '064201', '064202',           # Servicios de ambulancia
    # 07: Transporte (tasa 0% para pasajes)
    '073111', '073112',           # Tren
    '073121', '073122', '073123',  # Metro, tren ligero, trolebús
    '073211', '073212', '073213', '073214', '073215',  # Autobuses y colectivos
    '073221',                     # Taxi (puede tener 16% en algunos casos)
    '073222',                     # Servicios de transporte privado (Uber, etc.)
    '073230',                     # Transporte escolar
    '073290',                     # Otro transporte por carretera
    '073310', '073320',           # Vuelos nacionales e internacionales (pueden tener 16%)
    '073401', '073402',           # Transporte marítimo
    '073500',                     # Transporte combinado
    '073600',                     # Funicular, teleférico
    # 08: Comunicaciones (generalmente 16%)
    # 09: Recreación y cultura (tasa 0% para libros, periódicos, revistas)
    '097111',                     # Libros de texto
    '097191',                     # Libros de ficción
    '097210',                     # Periódicos
    '097220',                     # Revistas
    # 10: Educación (tasa 0% para servicios educativos)
    '101011', '101012', '101013', '101014', '101015',  # Preescolar
    '101021', '101022', '101023', '101024', '101025',  # Primaria
    '102001', '102002', '102003', '102004', '102005',  # Secundaria
    '103001', '103002', '103003', '103004', '103005',  # Bachillerato
    '104011', '104012', '104013', '104014',           # Carrera técnica
    '104015', '104016', '104017',                     # Educación superior
    '104018', '104019', '10401A',                     # Posgrado
    '105010',                     # Tutorías
    '105091', '105092', '105093',  # Educación para el trabajo
    # 11: Restaurantes y hoteles (generalmente 16%, pero algunos servicios pueden tener 0%)
    # 12: Seguros y servicios financieros (generalmente 16%)
    # 13: Cuidado personal (generalmente 16%)
    # 17: Gastos diversos (algunos pueden tener 0%)
    '173133', '173134',           # Materiales y servicios para construcción (pueden tener 16%)
    '173411',                     # Contribuciones para obras públicas (pueden tener 0%)
    '173416',                     # Ayuda a parientes (puede tener 0%)
    '173417',                     # Donativos (pueden tener 0%)
    # T: Regalos (la tasa depende del producto)
]

df_gastos_2024["lleva_iva"] = (
    ~df_gastos_2024["clave"].isin(CLAVES_IVA_CERO_2024)
).astype(int)

# ==========================================================
# Separar gasto gravado y no gravado
# ==========================================================

df_gastos_2024["gasto_iva"] = np.where(
    (df_gastos_2024["lleva_iva"] == 1) &
    (df_gastos_2024["es_formal"] == 1),
    df_gastos_2024["gasto_tri"],
    0
)

df_gastos_2024["gasto_no_iva"] = np.where(
    df_gastos_2024["lleva_iva"] == 0,
    df_gastos_2024["gasto_tri"],
    0
)

# ==========================================================
# Gasto en electricidad y en gasolina 
# ==========================================================

df_gastos_2024["gasto_gas"] = np.where(
    df_gastos_2024["clave"].isin([
        "072210",  # Diésel
        "072221",  # Gasolina magna, regular (bajo octanaje)
        "072222",  # Gasolina premium (alto octanaje)
        "072230"   # Otros combustibles: gas LP, electricidad, etc.
    ]),
    df_gastos_2024["gasto_tri"],
    0
)

df_gastos_2024["electricity"] = np.where(
    df_gastos_2024["clave"] == "045100",	#Energía eléctrica
    df_gastos_2024["gasto_tri"], 
    0
)


# ==========================================================
# IVA pagado (16% incluido en el precio)
# ==========================================================

FACTOR_IVA = 16 / 116

df_gastos_2024["iva_pagado"] = df_gastos_2024["gasto_iva"] * FACTOR_IVA

# ==========================================================
# Resumen por hogar
# ==========================================================

iva_hogar_2024 = (
    df_gastos_2024
    .groupby(["folioviv", "foliohog"], as_index=False)
    .agg(
        gasto_con_iva=("gasto_iva", "sum"),
        gasto_sin_iva=("gasto_no_iva", "sum"),
        iva_pagado=("iva_pagado", "sum"),
        gasto_total=("gasto_tri", "sum"),
        gasto_gas = ("gasto_gas", "sum"), 
        gasto_electricity = ("electricity", "sum")
    )
    .assign(year=2024)
)

iva_hogar_2024.head()

,folioviv,foliohog,gasto_con_iva,gasto_sin_iva,iva_pagado,gasto_total,gasto_gas,gasto_electricity,year
0,0100001901,1,30975.11,8344.67,4272.428966,39988.34,6967.74,195.0,2024
1,0100001902,1,26041.44,10427.04,3591.922759,36468.48,4354.83,900.0,2024
2,0100001904,1,16618.46,8644.97,2292.201379,28601.26,6967.74,663.0,2024
3,0100001905,1,30382.94,12812.31,4190.750345,43683.82,2903.22,274.5,2024
4,0100002501,1,82446.22,27668.50,11371.892414,124056.39,0.00,481.5,2024


In [6]:
df_final_2 = pd.concat([iva_hogar_2018, iva_hogar_2024], axis=0, ignore_index=True)

df_final_2.to_csv(PROCESSED_DIR / "iva_hogar.csv", index=False)
print("Saved iva_hogar.csv")

Saved iva_hogar.csv


# Educación y salud 

In [69]:
# ==========================================================
# Cargar base de gasto en salud
# ==========================================================

health = pd.read_excel(
    PROCESSED_DIR / "health.xlsx",
        dtype={
        "CVE_ENT": str
    }
)

health.head()

,CVE_ENT,ENTIDAD FEDERATIVA,YEAR,SIN SS,IMSS,ISSSTE,PEMEX
0,01,AGUASCALIENTES,2018,4711.83,3625.40,3581.75,1637.83
1,02,BAJA CALIFORNIA,2018,3536.70,3748.35,5496.69,888.73
2,03,BAJA CALIFORNIA SUR,2018,4768.90,5282.14,6252.78,0.00
3,04,CAMPECHE,2018,5530.83,4099.36,4123.39,26438.99
4,05,COAHUILA DE ZARAGOZA,2018,3853.22,3776.91,4008.33,2065.34


In [61]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_poblacion_2018= pd.read_csv(
    RAW_DIR / "poblacion_2018.csv",
    dtype={
        "folioviv": str,
        "foliohog": str
    },
    low_memory=False
)


# ==========================================================
# Education  
# ==========================================================

df_poblacion_2018["education"] = np.where(
    (df_poblacion_2018['asis_esc'] == "1") & (df_poblacion_2018['tipoesc'] == "1"),
    1,
    0
)


# ==========================================================
# Gasto público en educación (2024)
# ==========================================================

# Inicializar en cero
df_poblacion_2018["education_expenditure"] = 0.0

# Básica
mask_basica = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["1", "2", "3"]))
)

# prescolar
mask_prescolar = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["1"]))
)

# primaria
mask_primaria = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["2"]))
)

# secundaria
mask_secundaria = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["3"]))
)

# Profesional tecnico
mask_profesiona_tecnico = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["4", "5"]))
)

# Bachillerato
mask_bachillerato = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["6"]))
)

# Superior
mask_superior = (
    (df_poblacion_2018["asis_esc"] == "1") &
    (df_poblacion_2018["tipoesc"] == "1") &
    (df_poblacion_2018["nivel"].isin(["7", "8", "9"]))
)

# Asignar gasto por alumno (miles de pesos)
df_poblacion_2018.loc[mask_prescolar, "education_expenditure"] = 19.3
df_poblacion_2018.loc[mask_primaria, "education_expenditure"] = 17.6
df_poblacion_2018.loc[mask_secundaria, "education_expenditure"] = 27.0
df_poblacion_2018.loc[mask_profesiona_tecnico, "education_expenditure"] = 25.7
df_poblacion_2018.loc[mask_bachillerato, "education_expenditure"] = 36.9
df_poblacion_2018.loc[mask_superior, "education_expenditure"] = 82.7


# ==========================================================
# Health
# ==========================================================


df_poblacion_2018["health"] = np.where(
    (
        (df_poblacion_2018["inst_1"] == "1") |
        (df_poblacion_2018["inst_2"] == "2") |
        (df_poblacion_2018["inst_3"] == "3") |
        (df_poblacion_2018["inst_4"] == "4") |
        (df_poblacion_2018["inst_5"] == "5") |
        (df_poblacion_2018["segpop"] == "1")
    ),
    1,
    0
)


in_kind_transfers_2018 = df_poblacion_2018[["folioviv", "foliohog", "numren", "sexo", 
                                            "edad", "etnia", "education", "education_expenditure","health"]]

in_kind_transfers_2018["year"] = 2018

in_kind_transfers_2018.head()


,folioviv,foliohog,numren,sexo,edad,etnia,education,education_expenditure,health,year
0,0100013601,1,1,1,74,1,0,0.0,1,2018
1,0100013601,1,2,2,70,2,0,0.0,1,2018
2,0100013601,1,3,1,38,1,0,0.0,0,2018
3,0100013602,1,1,1,48,2,0,0.0,1,2018
4,0100013602,1,2,2,47,2,0,0.0,1,2018


In [63]:
df_poblacion_2018["nivel"].value_counts()

nivel
      191008
06     30543
07     14871
09     12631
01     10027
12      8991
13       543
10       241
08       223
11       125
04         3
Name: count, dtype: int64

In [62]:
in_kind_transfers_2018.describe()

,numren,sexo,edad,education,education_expenditure,health,year
count,269206.000000,269206.000000,269206.000000,269206.000000,269206.0,269206.000000,269206.0
mean,2.770633,1.510869,31.526378,0.261684,0.0,0.840122,2018.0
std,1.731784,0.499883,21.387297,0.439552,0.0,0.366493,0.0
min,1.000000,1.000000,0.000000,0.000000,0.0,0.000000,2018.0
25%,1.000000,1.000000,14.000000,0.000000,0.0,1.000000,2018.0
50%,2.000000,2.000000,28.000000,0.000000,0.0,1.000000,2018.0
75%,4.000000,2.000000,47.000000,1.000000,0.0,1.000000,2018.0
max,22.000000,2.000000,110.000000,1.000000,0.0,1.000000,2018.0


In [58]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_poblacion_2024= pd.read_csv(
    RAW_DIR / "poblacion_2024.csv",
    dtype={
        "folioviv": str,
        "foliohog": str
    },
    low_memory=False
)


# ==========================================================
# Education  
# ==========================================================

df_poblacion_2024["education"] = np.where(
    (df_poblacion_2024['asis_esc'] == 1) & (df_poblacion_2024['tipoesc'] == 1),
    1,
    0
)

# ==========================================================
# Gasto público en educación (2024)
# ==========================================================


# Inicializar en cero
df_poblacion_2024["education_expenditure"] = 0.0

# Básica
mask_basica = (
    (df_poblacion_2024["asis_esc"] == 1) &
    (df_poblacion_2024["tipoesc"] == 1) &
    (df_poblacion_2024["nivel"].isin([1, 2, 3]))
)

# Media superior
mask_media = (
    (df_poblacion_2024["asis_esc"] == 1) &
    (df_poblacion_2024["tipoesc"] == 1) &
    (df_poblacion_2024["nivel"].isin([4, 5]))
)

# Superior
mask_superior = (
    (df_poblacion_2024["asis_esc"] == 1) &
    (df_poblacion_2024["tipoesc"] == 1) &
    (df_poblacion_2024["nivel"].isin([6, 7, 8, 9]))
)

# Otros
mask_otros = (
    (df_poblacion_2024["asis_esc"] == 1) &
    (df_poblacion_2024["tipoesc"] == 1) &
    ~(mask_basica | mask_media | mask_superior)
)

# Asignar gasto por alumno (miles de pesos)
df_poblacion_2024.loc[mask_basica, "education_expenditure"] = 32.8
df_poblacion_2024.loc[mask_media, "education_expenditure"] = 31.3
df_poblacion_2024.loc[mask_superior, "education_expenditure"] = 64.0

# Repartir "Otros"
n_otros = mask_otros.sum()

#if n_otros > 0:
 #   otros_pc = (136_587.8 * 1_000_000) / n_otros   # pesos por persona
  #  otros_pc /= 1000                               # miles de pesos por persona
   # df_poblacion_2024.loc[mask_otros, "education_expenditure"] = otros_pc

# ==========================================================
# Health
# ==========================================================


df_poblacion_2024["health"] = np.where(
    (
        (df_poblacion_2024["inst_1"] == 1) |
        (df_poblacion_2024["inst_2"] == 2) |
        (df_poblacion_2024["inst_3"] == 3) |
        (df_poblacion_2024["inst_4"] == 4) |
        (df_poblacion_2024["inst_5"] == 5) |
        (df_poblacion_2024["inst_6"] == 6) 
    ),
    1,
    0
)

in_kind_transfers_2024 = df_poblacion_2024[["folioviv", "foliohog", "numren", "sexo", 
                                            "edad", "etnia", "education","education_expenditure" ,"health"]]

in_kind_transfers_2024["year"] = 2024

in_kind_transfers_2024.head()


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_77604/3771392400.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_poblacion_2024["education"] = np.where(
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_77604/3771392400.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_poblacion_2024["education_expenditure"] = 0.0
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_77604/3771392400.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.ins

,folioviv,foliohog,numren,sexo,edad,etnia,education,education_expenditure,health,year
0,0100001901,1,1,1,32,2.0,0,0.0,1,2024
1,0100001901,1,2,2,24,2.0,0,0.0,1,2024
2,0100001901,1,3,2,5,2.0,0,0.0,1,2024
3,0100001901,1,4,1,2,NaN,0,0.0,1,2024
4,0100001902,1,1,1,48,2.0,0,0.0,1,2024


In [59]:
in_kind_transfers_2024.describe()

,numren,sexo,edad,etnia,education,education_expenditure,health,year
count,308598.000000,308598.000000,308598.000000,297955.000000,308598.000000,308598.000000,308598.000000,308598.0
mean,2.644000,1.517923,33.881305,1.737917,0.240426,12.761016,0.633410,2024.0
std,1.645322,0.499679,21.779162,0.439768,0.427343,24.992345,0.481874,0.0
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,2024.0
25%,1.000000,1.000000,15.000000,1.000000,0.000000,0.000000,0.000000,2024.0
50%,2.000000,2.000000,32.000000,2.000000,0.000000,0.000000,1.000000,2024.0
75%,4.000000,2.000000,50.000000,2.000000,0.000000,0.000000,1.000000,2024.0
max,20.000000,2.000000,106.000000,2.000000,1.000000,64.000000,1.000000,2024.0


In [10]:
df_final_3 = pd.concat([in_kind_transfers_2018, in_kind_transfers_2024], axis=0, ignore_index=True)


df_final_3.to_csv(PROCESSED_DIR / "in_kind_transfers.csv", index=False)
print("Saved in_kind_transfers.csv")

Saved in_kind_transfers.csv


# Concentrado de hogar 

In [11]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_concentrado_2018= pd.read_csv(
    RAW_DIR / "concentradohogar_2018.csv",
    dtype={
        "folioviv": str,
        "foliohog": str
    },
    low_memory=False
)


# ==========================================================
# Demographic household 2018
# ==========================================================

demographic_household_2018 = (
    df_concentrado_2018[
        [
            "folioviv",
            "foliohog",
            "ubica_geo",
            "upm",
            "est_dis",
            "factor",
        ]
    ]
    .copy()
)

# Asegurar que ubica_geo tenga 5 dígitos
demographic_household_2018["ubica_geo"] = (
    demographic_household_2018["ubica_geo"]
    .astype(str)
    .str.zfill(5)
)

# Estado (primeros 2 dígitos)
demographic_household_2018["state"] = (
    demographic_household_2018["ubica_geo"]
    .str[:2]
)

# Municipio (últimos 3 dígitos)
demographic_household_2018["municipality"] = (
    demographic_household_2018["ubica_geo"]
    .str[2:]
)

# Reordenar columnas
demographic_household_2018 = demographic_household_2018[
    [
        "folioviv",
        "foliohog",
        "ubica_geo",
        "state",
        "municipality",
        "upm",
        "est_dis",
        "factor",
    ]
]

demographic_household_2018["year"] = 2018

demographic_household_2018.head()

,folioviv,foliohog,ubica_geo,state,municipality,upm,est_dis,factor,year
0,0100013601,1,01001,01,001,1,2,179,2018
1,0100013602,1,01001,01,001,1,2,179,2018
2,0100013603,1,01001,01,001,1,2,179,2018
3,0100013604,1,01001,01,001,1,2,179,2018
4,0100013606,1,01001,01,001,1,2,179,2018


In [12]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_concentrado_2024= pd.read_csv(
    RAW_DIR / "concentradohogar_2024.csv",
    dtype={
        "folioviv": str,
        "foliohog": str
    },
    low_memory=False
)

# ==========================================================
# Demographic household 2024
# ==========================================================

demographic_household_2024 = (
    df_concentrado_2024[
        [
            "folioviv",
            "foliohog",
            "ubica_geo",
            "upm",
            "est_dis",
            "factor",
        ]
    ]
    .copy()
)

# Asegurar que ubica_geo tenga 5 dígitos
demographic_household_2024["ubica_geo"] = (
    demographic_household_2024["ubica_geo"]
    .astype(str)
    .str.zfill(5)
)

# Estado (primeros 2 dígitos)
demographic_household_2024["state"] = (
    demographic_household_2024["ubica_geo"]
    .str[:2]
)

# Municipio (últimos 3 dígitos)
demographic_household_2024["municipality"] = (
    demographic_household_2024["ubica_geo"]
    .str[2:]
)

# Reordenar columnas
demographic_household_2024 = demographic_household_2024[
    [
        "folioviv",
        "foliohog",
        "ubica_geo",
        "state",
        "municipality",
        "upm",
        "est_dis",
        "factor",
    ]
]

demographic_household_2024["year"] = 2024

demographic_household_2024.head()

,folioviv,foliohog,ubica_geo,state,municipality,upm,est_dis,factor,year
0,0100001901,1,01001,01,001,1,1,207,2024
1,0100001902,1,01001,01,001,1,1,207,2024
2,0100001904,1,01001,01,001,1,1,207,2024
3,0100001905,1,01001,01,001,1,1,207,2024
4,0100002501,1,01001,01,001,2,2,196,2024


In [13]:
df_final_4 = pd.concat([demographic_household_2018, demographic_household_2024], axis=0, ignore_index=True)


df_final_4.to_csv(PROCESSED_DIR / "demographic_household.csv", index=False)
print("Saved in_kind_transdemographic_household_2024fers.csv")

Saved in_kind_transdemographic_household_2024fers.csv
